In [ ]:
# Cell 1 - GFM flood pipeline inputs and run controls
from __future__ import annotations

from pathlib import Path
import json

import pandas as pd

# USER INPUTS
ISO3 = "MLI"
AS_OF_DATE = "2025-12-31"  # YYYY-MM-DD
LOOKBACK_MONTHS = 12

# INPUTS
GDB_ZIP_PATH = Path("./data/cod-ab/global_admin_boundaries_matched_latest.gdb.zip")
ADMIN2_LAYER = "admin2"
worldpop_name = f"{ISO3.lower()}_pop_2025_CN_100m_R2025A_v1.tif"
WORLDPOP_PATH = Path(f"./data/population/{worldpop_name}")

# GFM STAC SETTINGS
STAC_API_URL = "https://stac.eodc.eu/api/v1"
GFM_COLLECTION_ID = "GFM"
GFM_ASSET_KEY = "ensemble_flood_extent"

# OUTPUT SETTINGS
OUT_ROOT = Path("./outputs")
PIPELINE_NAME = "gfm_flood"
PRECHECK_MAKE_PLOTS = True


def _assert_exists(path: Path, label: str) -> None:
    if not path.exists():
        raise FileNotFoundError(f"Missing {label}: {path.resolve()}")


_assert_exists(GDB_ZIP_PATH, "admin boundaries GDB zip")
_assert_exists(WORLDPOP_PATH, "WorldPop raster")
AS_OF_DT = pd.to_datetime(AS_OF_DATE)
if pd.isna(AS_OF_DT):
    raise ValueError(f"AS_OF_DATE must be YYYY-MM-DD, got: {AS_OF_DATE}")

print("ISO3:", ISO3)
print("AS_OF_DATE:", AS_OF_DATE)
print("LOOKBACK_MONTHS:", LOOKBACK_MONTHS)
print("WORLDPOP_PATH:", WORLDPOP_PATH.resolve())

# Preflight coverage thresholds
WORLDPOP_COVERAGE_MIN_PCT = 98.0
FLOOD_STAC_COVERAGE_MIN_PCT = 70


# Flood severity/exposure controls
FLOOD_BINARY_THRESHOLD_DAYS = 0  # binary exposure uses flood_days > threshold

# Performance defaults for flood stack reduction
FLOOD_LOAD_CHUNKS = {"time": 14, "y": 512, "x": 512}


In [ ]:
# Cell 2 - Canonical directory scaffold and schema-compliant run metadata
from pathlib import Path
from datetime import datetime
import json

from wia_pipelines.config import RunConfig, initialize_run_metadata, validate_run_metadata


def ensure_dir(path: Path) -> Path:
    path.mkdir(parents=True, exist_ok=True)
    return path


RUN_CONFIG = RunConfig(
    hazard="flood",
    iso3=ISO3,
    as_of_date=AS_OF_DATE,
    lookback_months=LOOKBACK_MONTHS,
    output_root=OUT_ROOT,
    target_adm_level=2,
    buffer_km=0.0,
)

RUN_ID = RUN_CONFIG.run_id
WINDOW_START = pd.Timestamp(RUN_CONFIG.window_start.isoformat())
WINDOW_END = pd.Timestamp(RUN_CONFIG.window_end.isoformat())
BASE_DIR = ensure_dir(OUT_ROOT / "flood" / ISO3 / RUN_ID)

# Shared canonical contract directories + backward-compatible aliases
DIRS = {
    "base": BASE_DIR,
    "raw": ensure_dir(BASE_DIR / "raw"),
    "intermediate": ensure_dir(BASE_DIR / "intermediate"),
    "rasters": ensure_dir(BASE_DIR / "rasters"),
    "tables": ensure_dir(BASE_DIR / "tables"),
    "qc": ensure_dir(BASE_DIR / "qc"),
    "logs": ensure_dir(BASE_DIR / "logs"),
    "cache": ensure_dir(OUT_ROOT / "_cache" / "flood" / ISO3),
    "flood_native": ensure_dir(BASE_DIR / "rasters" / "flood_native"),
    "flood_worldpop": ensure_dir(BASE_DIR / "rasters" / "flood_worldpop"),
    "pop_exposed": ensure_dir(BASE_DIR / "rasters" / "pop_exposed"),
}

RUN_METADATA_PATH = DIRS["base"] / "run_metadata.json"

run_metadata = initialize_run_metadata(
    RUN_CONFIG,
    paths={
        "base": DIRS["base"],
        "raw": DIRS["raw"],
        "rasters": DIRS["rasters"],
        "tables": DIRS["tables"],
        "qc": DIRS["qc"],
        "logs": DIRS["logs"],
        "cache": DIRS["cache"],
    },
)

run_metadata["pipeline"] = PIPELINE_NAME
run_metadata["flood_config"] = {
    "collection_id": GFM_COLLECTION_ID,
    "asset_key": GFM_ASSET_KEY,
    "stac_api_url": STAC_API_URL,
    "precheck_make_plots": PRECHECK_MAKE_PLOTS,
}

# Keep legacy key alias for compatibility with earlier flood notebook logic.
run_metadata.setdefault("artifacts", [])
run_metadata["artefacts"] = run_metadata["artifacts"]


def _write_metadata() -> None:
    run_metadata["artefacts"] = run_metadata.get("artifacts", [])
    validate_run_metadata(run_metadata)
    RUN_METADATA_PATH.write_text(json.dumps(run_metadata, indent=2, default=str))


def update_run_metadata_artifact(kind: str, path: Path, note: str | None = None) -> None:
    artifacts = run_metadata.setdefault("artifacts", [])
    artifacts.append(
        {
            "timestamp": datetime.utcnow().isoformat(timespec="seconds") + "Z",
            "kind": kind,
            "path": str(path),
            "note": note,
        }
    )
    _write_metadata()


_write_metadata()

print("RUN_ID:", RUN_ID)
print("BASE_DIR:", BASE_DIR.resolve())
print("RUN_METADATA_PATH:", RUN_METADATA_PATH)
print("DIRS:", sorted(DIRS.keys()))

In [ ]:
# Cell 3 - Admin2 AOI and bounds hash (no artificial AOI buffer)
import hashlib
import geopandas as gpd
from shapely.geometry import mapping


gdf = gpd.read_file(GDB_ZIP_PATH, layer=ADMIN2_LAYER)
if "iso3" not in gdf.columns:
    raise KeyError(f"'iso3' column not found in admin layer. Columns: {list(gdf.columns)}")

admin2_gdf = gdf[gdf["iso3"] == ISO3].copy()
if admin2_gdf.empty:
    raise ValueError(f"No admin2 features found for ISO3={ISO3}")
if "adm2_pcode" not in admin2_gdf.columns:
    raise KeyError(f"'adm2_pcode' not found in admin layer. Columns: {list(admin2_gdf.columns)}")
if admin2_gdf.crs is None:
    raise ValueError("Admin2 CRS is missing; cannot continue.")

admin2_4326 = admin2_gdf.to_crs("EPSG:4326")
country_geom_4326 = admin2_4326.geometry.union_all()
if country_geom_4326.is_empty:
    raise RuntimeError(f"Empty AOI geometry for ISO3={ISO3}")

west, south, east, north = map(float, country_geom_4326.bounds)
stac_bbox = [west, south, east, north]
stac_intersects_geom = mapping(country_geom_4326)

bounds_hash_payload = {
    "iso3": ISO3,
    "layer": ADMIN2_LAYER,
    "bounds_wsen": [west, south, east, north],
}
bounds_hash = hashlib.sha256(
    json.dumps(bounds_hash_payload, sort_keys=True).encode("utf-8")
).hexdigest()

bounds_hash_path = DIRS["logs"] / f"{ISO3}_bounds_hash.txt"
bounds_hash_path.write_text(bounds_hash)
update_run_metadata_artifact("aoi_bounds_hash", bounds_hash_path, "Admin2 union bounds hash")

run_metadata["aoi"] = {
    "bounds_epsg4326": {"west": west, "south": south, "east": east, "north": north},
    "stac_bbox": stac_bbox,
    "bounds_hash": bounds_hash,
    "bounds_hash_path": str(bounds_hash_path),
}
# Keep legacy keys consumed by older downstream cells.
run_metadata["aoi_bounds_epsg4326"] = run_metadata["aoi"]["bounds_epsg4326"]
run_metadata["bounds_hash"] = bounds_hash
_write_metadata()

print("Country bounds (W,S,E,N):", stac_bbox)
print("Bounds hash:", bounds_hash)

In [ ]:
# Cell 4 - STAC client setup and collection metadata capture
from pystac_client import Client


def _json_safe(obj):
    import datetime as _dt
    import numpy as _np
    import pandas as _pd

    if obj is None:
        return None
    if isinstance(obj, (_dt.datetime, _dt.date, _pd.Timestamp)):
        return obj.isoformat()
    if isinstance(obj, (_np.integer,)):
        return int(obj)
    if isinstance(obj, (_np.floating,)):
        return float(obj)
    if isinstance(obj, dict):
        return {str(k): _json_safe(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_json_safe(v) for v in obj]
    return obj


stac_client = Client.open(STAC_API_URL)
collection = stac_client.get_collection(GFM_COLLECTION_ID)

run_metadata["gfm_collection"] = _json_safe(
    {
        "id": collection.id,
        "title": collection.title,
        "description": collection.description,
        "license": collection.license,
        "spatial_extent": collection.extent.spatial.bboxes if collection.extent else None,
        "temporal_extent": collection.extent.temporal.intervals if collection.extent else None,
    }
)
_write_metadata()

print("Connected to STAC:", STAC_API_URL)
print("Collection:", collection.id)

In [ ]:
# Cell 5 - STAC item search in run window and manifest capture
from datetime import timezone


start_dt = (
    AS_OF_DT - pd.DateOffset(months=LOOKBACK_MONTHS) + pd.Timedelta(days=1)
).to_pydatetime()
end_dt = AS_OF_DT.to_pydatetime()
start_dt = start_dt.replace(tzinfo=timezone.utc)
end_dt = end_dt.replace(tzinfo=timezone.utc)
datetime_range = (
    f"{start_dt.isoformat().replace('+00:00', 'Z')}/"
    f"{end_dt.isoformat().replace('+00:00', 'Z')}"
)

manifest_key = __import__("hashlib").sha256(
    json.dumps(
        {
            "iso3": ISO3,
            "bounds_hash": bounds_hash,
            "collection": GFM_COLLECTION_ID,
            "asset": GFM_ASSET_KEY,
            "window_start": str(pd.to_datetime(start_dt).date()),
            "window_end": str(pd.to_datetime(end_dt).date()),
        },
        sort_keys=True,
    ).encode("utf-8")
).hexdigest()
manifest_json = DIRS["logs"] / f"{ISO3}_stac_manifest_{manifest_key}.json"
manifest_csv = DIRS["logs"] / f"{ISO3}_stac_manifest_{manifest_key}.csv"

search = stac_client.search(
    collections=[GFM_COLLECTION_ID],
    datetime=datetime_range,
    intersects=stac_intersects_geom,
    max_items=None,
)
items = list(search.items())
if not items:
    raise RuntimeError(
        f"No STAC items returned for {ISO3} in {datetime_range} for collection={GFM_COLLECTION_ID}."
    )

rows = []
for item in items:
    rows.append(
        {
            "id": item.id,
            "datetime": item.datetime.isoformat() if item.datetime else None,
            "collection": item.collection_id,
            "bbox": item.bbox,
            "asset_keys": ",".join(sorted(item.assets.keys())),
        }
    )

items_df = pd.DataFrame(rows).sort_values("datetime")

# Filter to items carrying the flood asset, overlapping AOI bbox, and unique asset href.
from shapely.geometry import box as _sbox

admin_box = _sbox(*stac_bbox)
filtered_items = []
seen_hrefs = set()
dropped_no_asset = 0
dropped_no_bbox = 0
dropped_no_overlap = 0
dropped_duplicate_href = 0

for item in items:
    asset = item.assets.get(GFM_ASSET_KEY)
    if asset is None or not getattr(asset, "href", None):
        dropped_no_asset += 1
        continue
    if item.bbox is None:
        dropped_no_bbox += 1
        continue
    ibox = _sbox(*item.bbox)
    if not admin_box.intersects(ibox):
        dropped_no_overlap += 1
        continue
    href = str(asset.href)
    if href in seen_hrefs:
        dropped_duplicate_href += 1
        continue
    seen_hrefs.add(href)
    filtered_items.append(item)

if not filtered_items:
    raise RuntimeError(
        f"No usable STAC items remain after filtering/dedup for {ISO3}."
    )

items = filtered_items

manifest = {
    "iso3": ISO3,
    "collection": GFM_COLLECTION_ID,
    "asset_key": GFM_ASSET_KEY,
    "bounds_hash": bounds_hash,
    "window_start": str(pd.to_datetime(start_dt).date()),
    "window_end": str(pd.to_datetime(end_dt).date()),
    "n_items": int(len(items)),
    "items": items_df.to_dict(orient="records"),
}
manifest_json.write_text(json.dumps(manifest, indent=2, default=str))
items_df.to_csv(manifest_csv, index=False)
update_run_metadata_artifact("stac_manifest_json", manifest_json, "STAC search manifest")
update_run_metadata_artifact("stac_manifest_csv", manifest_csv, "STAC search table")

run_metadata["stac_search"] = {
    "datetime_range": datetime_range,
    "n_items": int(len(items)),
    "manifest_key": manifest_key,
    "filtering": {
        "dropped_no_asset": int(dropped_no_asset),
        "dropped_no_bbox": int(dropped_no_bbox),
        "dropped_no_overlap": int(dropped_no_overlap),
        "dropped_duplicate_href": int(dropped_duplicate_href),
    },
}
_write_metadata()

asset_coverage = items_df["asset_keys"].str.contains(GFM_ASSET_KEY, regex=False).mean()
print("STAC datetime:", datetime_range)
print("Items found after filtering:", len(items))
print("Dropped items:", {"no_asset": dropped_no_asset, "no_bbox": dropped_no_bbox, "no_overlap": dropped_no_overlap, "duplicate_href": dropped_duplicate_href})
print(f"Items containing '{GFM_ASSET_KEY}': {asset_coverage:.0%}")

In [ ]:
# Cell 6 - Preflight coverage checks (fail fast before heavy flood raster load)
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd
from shapely.geometry import box
from shapely.ops import unary_union

from wia_pipelines.hazards.coverage_checks import check_worldpop_coverage


admin_bounds_wsen = (west, south, east, north)
worldpop_cov = check_worldpop_coverage(admin_bounds_wsen, WORLDPOP_PATH)

admin_box = box(*admin_bounds_wsen)
item_bboxes = []
item_coverages = []
item_ids = []
for item in items:
    if item.bbox is None or GFM_ASSET_KEY not in item.assets:
        continue
    ibox = box(*item.bbox)
    item_bboxes.append(ibox)
    item_ids.append(item.id)
    inter = admin_box.intersection(ibox)
    cov = 0.0 if admin_box.area == 0 else float((inter.area / admin_box.area) * 100.0)
    item_coverages.append(cov)

if not item_bboxes:
    raise RuntimeError(
        f"No STAC items with '{GFM_ASSET_KEY}' and bbox metadata were returned for {ISO3}."
    )

union_bbox = unary_union(item_bboxes)
union_inter = admin_box.intersection(union_bbox)
union_cov = 0.0 if admin_box.area == 0 else float((union_inter.area / admin_box.area) * 100.0)
union_full = union_cov >= FLOOD_STAC_COVERAGE_MIN_PCT
worldpop_ok = float(worldpop_cov.get("coverage_pct", 0.0)) >= WORLDPOP_COVERAGE_MIN_PCT

preflight = {
    "worldpop": worldpop_cov,
    "thresholds": {
        "worldpop_coverage_min_pct": float(WORLDPOP_COVERAGE_MIN_PCT),
        "flood_stac_union_coverage_min_pct": float(FLOOD_STAC_COVERAGE_MIN_PCT),
    },
    "flood_stac": {
        "item_count": len(item_bboxes),
        "item_ids": item_ids,
        "item_bbox_coverages_pct": [float(v) for v in item_coverages],
        "union_bbox_coverage_pct": float(union_cov),
        "union_full_coverage": bool(union_full),
        "datetime_range": datetime_range,
    },
}

plot_path = DIRS["qc"] / f"{ISO3}_flood_preflight_coverage.png"
if PRECHECK_MAKE_PLOTS:
    fig, ax = plt.subplots(figsize=(10, 8))
    admin2_gdf.to_crs("EPSG:4326").boundary.plot(ax=ax, linewidth=0.6, color="black")
    gpd.GeoSeries([admin_box], crs="EPSG:4326").boundary.plot(
        ax=ax, linewidth=1.2, color="red", linestyle="--"
    )

    union_gs = gpd.GeoSeries([union_bbox], crs="EPSG:4326")
    union_gs.boundary.plot(ax=ax, linewidth=1.0, color="royalblue")

    ax.set_title(f"{ISO3} Flood Preflight: STAC bbox union + Admin2 AOI")
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    fig.tight_layout()
    fig.savefig(plot_path, dpi=200)
    plt.close(fig)
    update_run_metadata_artifact("preflight_flood_plot", plot_path, "Flood STAC bbox union preflight")

run_metadata["preflight_coverage"] = preflight
_write_metadata()

print("WorldPop coverage pct:", f"{worldpop_cov['coverage_pct']:.3f}")
print("Flood STAC union coverage pct:", f"{union_cov:.3f}")
if PRECHECK_MAKE_PLOTS:
    print("Preflight plot:", plot_path)

if not worldpop_ok:
    raise RuntimeError(
        f"WorldPop coverage below threshold for {ISO3}: "
        f"{worldpop_cov['coverage_pct']:.3f}% < {WORLDPOP_COVERAGE_MIN_PCT:.3f}%."
    )
if not union_full:
    raise RuntimeError(
        f"Flood STAC item bbox union coverage below threshold for {ISO3}: "
        f"{union_cov:.3f}% < {FLOOD_STAC_COVERAGE_MIN_PCT:.3f}%."
    )

In [ ]:
# Cell 7 - Streaming GFM processing: compute flood_days severity with live progress/status monitoring
import json
import time

import numpy as np
import rasterio
import rioxarray  # noqa: F401
import xarray as xr
from rasterio.windows import from_bounds
from odc.stac import load as stac_load
from odc.geo.geobox import GeoBox


manifest_hash = manifest_key
status_path = DIRS["logs"] / f"{ISO3}_cell7_status.json"
start_ts = time.perf_counter()

# Build WorldPop-aligned intersection window from AOI bounds
with rasterio.open(WORLDPOP_PATH) as wp:
    wp_crs = wp.crs
    wp_res = wp.res
    wp_bounds = wp.bounds

win_left = max(west, float(wp_bounds.left))
win_right = min(east, float(wp_bounds.right))
win_bottom = max(south, float(wp_bounds.bottom))
win_top = min(north, float(wp_bounds.top))

if (win_left >= win_right) or (win_bottom >= win_top):
    raise RuntimeError("Admin bounds do not intersect WorldPop bounds.")

with rasterio.open(WORLDPOP_PATH) as wp:
    w = from_bounds(win_left, win_bottom, win_right, win_top, transform=wp.transform)
    w = w.round_offsets().round_lengths()
    w = w.intersection(rasterio.windows.Window(0, 0, wp.width, wp.height))
    win_width = int(w.width)
    win_height = int(w.height)

bbox_win = (win_left, win_bottom, win_right, win_top)
res = float(wp_res[0])
geobox_wp = GeoBox.from_bbox(bbox=bbox_win, crs=wp_crs, resolution=res, tight=True)

# Streaming accumulator state
n_total = len(items)
if n_total == 0:
    raise RuntimeError("No STAC items available for flood processing.")

flood_days_accum = None
template_da = None
flood_nodata = 255
processed = 0
success = 0
failed = 0
failures = []
progress_every = max(1, min(25, n_total // 20 if n_total > 20 else 1))


def write_status() -> None:
    elapsed = time.perf_counter() - start_ts
    rate = processed / elapsed if elapsed > 0 else 0.0
    remaining = max(0, n_total - processed)
    eta_seconds = remaining / rate if rate > 0 else None
    payload = {
        "iso3": ISO3,
        "stage": "cell7_streaming_flood_days",
        "processed": int(processed),
        "success": int(success),
        "failed": int(failed),
        "total": int(n_total),
        "pct_complete": float((processed / n_total) * 100.0),
        "elapsed_seconds": float(round(elapsed, 2)),
        "rate_items_per_second": float(round(rate, 4)),
        "eta_seconds": None if eta_seconds is None else float(round(eta_seconds, 2)),
        "latest_failures": failures[-5:],
    }
    status_path.write_text(json.dumps(payload, indent=2))


print(f"Cell 7 streaming start: {n_total} items")
print("Status file:", status_path)
write_status()

for idx, item in enumerate(items, start=1):
    processed = idx
    item_id = getattr(item, "id", f"item_{idx}")
    try:
        ds_i = stac_load(
            [item],
            bands=[GFM_ASSET_KEY],
            geobox=geobox_wp,
            chunks={"y": int(FLOOD_LOAD_CHUNKS["y"]), "x": int(FLOOD_LOAD_CHUNKS["x"])},
            dtype="uint8",
            fail_on_error=False,
        )

        if GFM_ASSET_KEY in ds_i.data_vars:
            da_i = ds_i[GFM_ASSET_KEY]
        elif len(ds_i.data_vars) > 0:
            fallback_var = list(ds_i.data_vars)[0]
            da_i = ds_i[fallback_var]
        else:
            raise RuntimeError("No data variables returned by stac_load for this item.")

        if "time" in da_i.dims:
            da_i = da_i.isel(time=0, drop=True)

        if template_da is None:
            template_da = da_i
            flood_nodata = int(da_i.attrs.get("nodata", 255))

        hit_i = ((da_i != flood_nodata) & (da_i > 0)).astype("uint16")
        hit_np = np.asarray(hit_i.compute().values, dtype="uint16")

        if flood_days_accum is None:
            flood_days_accum = np.zeros(hit_np.shape, dtype="uint16")

        flood_days_accum += hit_np
        success += 1

    except Exception as exc:
        failed += 1
        failures.append({"item_id": str(item_id), "error": str(exc)})

    if (idx % progress_every == 0) or (idx == n_total):
        write_status()
        elapsed = time.perf_counter() - start_ts
        rate = processed / elapsed if elapsed > 0 else 0.0
        eta = (n_total - processed) / rate if rate > 0 else float("nan")
        print(
            f"Progress {processed}/{n_total} ({(processed/n_total)*100:.1f}%) "
            f"success={success} failed={failed} rate={rate:.3f} it/s eta={eta/60:.1f} min"
        )

if flood_days_accum is None or template_da is None:
    raise RuntimeError("Flood streaming failed: no items were successfully processed.")

# Build xarray raster from accumulated severity counts and preserve geospatial metadata.
flood_days = xr.DataArray(
    flood_days_accum,
    dims=template_da.dims,
    coords={d: template_da.coords[d] for d in template_da.dims if d in template_da.coords},
    name="flood_days",
)
flood_days = flood_days.rio.write_crs(template_da.rio.crs, inplace=False)
try:
    flood_days = flood_days.rio.write_transform(template_da.rio.transform(), inplace=False)
except Exception:
    pass

# Derived binary mask for exposure. Default is flood_days > 0.
flood_binary = (flood_days > int(FLOOD_BINARY_THRESHOLD_DAYS)).astype("uint8")

elapsed_total = time.perf_counter() - start_ts
run_metadata["gfm_load"] = {
    "manifest_hash": str(manifest_hash),
    "worldpop_window_bounds_epsg4326": {
        "west": float(win_left),
        "south": float(win_bottom),
        "east": float(win_right),
        "north": float(win_top),
    },
    "worldpop_window_shape": [int(win_height), int(win_width)],
    "chunks": {k: int(v) for k, v in FLOOD_LOAD_CHUNKS.items()},
    "flood_nodata": int(flood_nodata),
    "severity_metric": "flood_days_sum",
    "binary_threshold_days": int(FLOOD_BINARY_THRESHOLD_DAYS),
    "streaming_mode": True,
    "items_total": int(n_total),
    "items_success": int(success),
    "items_failed": int(failed),
    "elapsed_seconds": float(round(elapsed_total, 2)),
    "status_file": str(status_path),
    "failures_preview": failures[-10:],
}
_write_metadata()

print("WorldPop-aligned load window:", (win_left, win_bottom, win_right, win_top))
print("Geobox shape:", geobox_wp.shape)
print("Flood nodata:", flood_nodata)
print("flood_days dims:", dict(flood_days.sizes))
print("Binary threshold (days):", FLOOD_BINARY_THRESHOLD_DAYS)
print(f"Cell 7 complete in {elapsed_total/60:.1f} minutes; success={success}, failed={failed}")

# Persist severity + binary rasters for robust downstream reruns.
MASK_DIR = ensure_dir(DIRS["flood_native"] / "flood")
FLOOD_DAYS_NAME = f"{ISO3}_flood_days_{WINDOW_START:%Y-%m-%d}_{WINDOW_END:%Y-%m-%d}"
FLOOD_MASK_NAME = f"{ISO3}_flood_any_{WINDOW_START:%Y-%m-%d}_{WINDOW_END:%Y-%m-%d}"

flood_days_tif = MASK_DIR / f"{FLOOD_DAYS_NAME}.tif"
out_tif = MASK_DIR / f"{FLOOD_MASK_NAME}.tif"  # legacy name kept for compatibility

flood_days.rio.write_nodata(0, inplace=False).rio.to_raster(
    flood_days_tif,
    compress="deflate",
    tiled=True,
    BIGTIFF="YES",
    blockxsize=512,
    blockysize=512,
    predictor=2,
)

flood_binary.rio.write_nodata(0, inplace=False).rio.to_raster(
    out_tif,
    compress="deflate",
    tiled=True,
    BIGTIFF="YES",
    blockxsize=512,
    blockysize=512,
    predictor=2,
)

update_run_metadata_artifact(
    "flood_days_tif",
    flood_days_tif,
    "Flood severity raster: number of flooded days per pixel",
)
update_run_metadata_artifact(
    "flood_mask_tif",
    out_tif,
    f"Binary flood mask derived from flood_days > {int(FLOOD_BINARY_THRESHOLD_DAYS)}",
)

run_metadata.setdefault("flood_mask", {})
run_metadata["flood_mask"].update(
    {
        "days_tif": str(flood_days_tif),
        "mask_tif": str(out_tif),
        "binary_threshold_days": int(FLOOD_BINARY_THRESHOLD_DAYS),
    }
)
_write_metadata()

print("Wrote flood days raster:", flood_days_tif)
print("Wrote flood binary mask:", out_tif)

In [ ]:
# Cell 8 — Population affected raster from flood_days-derived binary mask
# Output:
#   - pop_affected_flood_any.tif (float32; population counts where flood_days > threshold)
#   - QC PNG: WorldPop + pop_affected overlay

import json
import numpy as np
import rasterio
import rasterio.warp
import matplotlib.pyplot as plt
import rioxarray  # noqa: F401
from affine import Affine

# ---- Paths / names ----
POP_NAME = f"{ISO3}_pop_affected_flood_any_{WINDOW_START:%Y-%m-%d}_{WINDOW_END:%Y-%m-%d}"
POP_DIR = ensure_dir(DIRS["pop_exposed"] / "flood")
QC_DIR = ensure_dir(DIRS["qc"] / "flood")
MASK_DIR = ensure_dir(DIRS["flood_native"] / "flood")

POP_TIF_PATH = POP_DIR / f"{POP_NAME}.tif"
QC_PNG_PATH = QC_DIR / f"{POP_NAME}_preview.png"
FLOOD_DAYS_NAME = f"{ISO3}_flood_days_{WINDOW_START:%Y-%m-%d}_{WINDOW_END:%Y-%m-%d}"
FLOOD_MASK_NAME = f"{ISO3}_flood_any_{WINDOW_START:%Y-%m-%d}_{WINDOW_END:%Y-%m-%d}"

flood_days_tif = MASK_DIR / f"{FLOOD_DAYS_NAME}.tif"
out_tif = MASK_DIR / f"{FLOOD_MASK_NAME}.tif"  # legacy binary path

# ---- Load WorldPop as reference grid ----
with rasterio.open(WORLDPOP_PATH) as wp:
    wp_arr = wp.read(1).astype("float32")
    wp_transform = wp.transform
    wp_crs = wp.crs
    wp_shape = (wp.height, wp.width)
    wp_nodata = wp.nodata
    wp_bounds = wp.bounds

pop_valid = np.isfinite(wp_arr)
if wp_nodata is not None:
    pop_valid &= wp_arr != float(wp_nodata)
pop_valid &= wp_arr >= 0

# ---- Resolve flood severity/binary rasters ----
if not flood_days_tif.exists():
    if "flood_days" not in globals():
        raise RuntimeError(
            "Flood days raster is missing and flood_days is not in memory. "
            "Run Cell 7 before this cell."
        )
    flood_days.rio.write_nodata(0, inplace=False).rio.to_raster(
        flood_days_tif,
        compress="deflate",
        tiled=True,
        BIGTIFF="YES",
        blockxsize=512,
        blockysize=512,
        predictor=2,
    )
    update_run_metadata_artifact("flood_days_tif", flood_days_tif, "Persisted from in-memory flood_days")

with rasterio.open(flood_days_tif) as src_days:
    flood_days_arr = src_days.read(1).astype("uint16")
    flood_days_crs = src_days.crs
    flood_days_transform = src_days.transform
    flood_days_shape = (src_days.height, src_days.width)
    flood_days_bounds = src_days.bounds

same_shape = flood_days_shape == wp_shape
same_crs = str(flood_days_crs) == str(wp_crs)
same_transform = (
    isinstance(flood_days_transform, Affine)
    and isinstance(wp_transform, Affine)
    and flood_days_transform.almost_equals(wp_transform, precision=1e-9)
)
aligned = same_shape and same_crs and same_transform

if not aligned:
    print("Flood days raster not aligned to WorldPop grid; reprojecting nearest.")
    flood_days_reproj = np.zeros(wp_shape, dtype="uint16")
    rasterio.warp.reproject(
        source=flood_days_arr,
        destination=flood_days_reproj,
        src_transform=flood_days_transform,
        src_crs=flood_days_crs,
        dst_transform=wp_transform,
        dst_crs=wp_crs,
        resampling=rasterio.warp.Resampling.nearest,
        src_nodata=None,
        dst_nodata=0,
    )
    flood_days_arr = flood_days_reproj

# Binary exposure mask derived from severity raster.
flood_mask_arr = (flood_days_arr > int(FLOOD_BINARY_THRESHOLD_DAYS)).astype("uint8")
flooded = flood_mask_arr == 1

# Keep binary mask path available for downstream compatibility.
if not out_tif.exists():
    profile_mask = {
        "driver": "GTiff",
        "height": wp_shape[0],
        "width": wp_shape[1],
        "count": 1,
        "dtype": "uint8",
        "crs": wp_crs,
        "transform": wp_transform,
        "nodata": 0,
        "compress": "deflate",
        "tiled": True,
        "blockxsize": 512,
        "blockysize": 512,
    }
    with rasterio.open(out_tif, "w", **profile_mask) as dst:
        dst.write(flood_mask_arr, 1)

# ---- Compute population affected ----
pop_affected = np.zeros(wp_shape, dtype="float32")
pop_affected[flooded & pop_valid] = wp_arr[flooded & pop_valid]

# Optional severity-weighted exposure output.
pop_weighted_days = np.zeros(wp_shape, dtype="float32")
pop_weighted_days[pop_valid] = wp_arr[pop_valid] * flood_days_arr[pop_valid].astype("float32")
POP_WEIGHTED_DAYS_TIF = POP_DIR / f"{ISO3}_pop_weighted_flood_days_{WINDOW_START:%Y-%m-%d}_{WINDOW_END:%Y-%m-%d}.tif"

# ---- QC stats ----
total_pop = float(wp_arr[pop_valid].sum())
affected_pop = float(pop_affected.sum())
affected_pct = (affected_pop / total_pop) * 100.0 if total_pop > 0 else np.nan

affected_cells = int((pop_affected > 0).sum())
valid_cells = int(pop_valid.sum())
flooded_cells = int(flooded.sum())
mean_flood_days_flooded = float(flood_days_arr[flooded].mean()) if flooded_cells > 0 else 0.0
max_flood_days = int(flood_days_arr.max())

print("WorldPop nodata:", wp_nodata)
print("Flood binary threshold days:", FLOOD_BINARY_THRESHOLD_DAYS)
print("Flood days max:", max_flood_days)
print("Mean flood days (flooded cells):", f"{mean_flood_days_flooded:.2f}")
print("WorldPop total pop (valid cells):", f"{total_pop:,.0f}")
print("Flood-affected pop:", f"{affected_pop:,.0f}", f"({affected_pct:.3f}%)")
print(
    f"Cells: valid_pop={valid_cells:,} flooded_mask={flooded_cells:,} pop_affected_nonzero={affected_cells:,}"
)

# ---- Write rasters ----
profile_pop = {
    "driver": "GTiff",
    "height": wp_shape[0],
    "width": wp_shape[1],
    "count": 1,
    "dtype": "float32",
    "crs": wp_crs,
    "transform": wp_transform,
    "nodata": 0.0,
    "compress": "deflate",
    "tiled": True,
    "blockxsize": 512,
    "blockysize": 512,
}

with rasterio.open(POP_TIF_PATH, "w", **profile_pop) as dst:
    dst.write(pop_affected, 1)
with rasterio.open(POP_WEIGHTED_DAYS_TIF, "w", **profile_pop) as dst:
    dst.write(pop_weighted_days, 1)

print("Wrote pop affected raster:", POP_TIF_PATH)
print("Wrote pop weighted flood-days raster:", POP_WEIGHTED_DAYS_TIF)

# ---- QC figure ----
ds = 8
wp_plot = np.where(pop_valid, wp_arr, np.nan)[::ds, ::ds]
pa_plot = np.where(pop_affected > 0, pop_affected, np.nan)[::ds, ::ds]

left, bottom, right, top = rasterio.transform.array_bounds(wp_shape[0], wp_shape[1], wp_transform)

fig, ax = plt.subplots(figsize=(10, 8))
ax.imshow(wp_plot, extent=(left, right, bottom, top), origin="upper", vmin=0)
ax.imshow(pa_plot, extent=(left, right, bottom, top), origin="upper", alpha=0.65, vmin=0)
admin2_gdf.to_crs("EPSG:4326").boundary.plot(ax=ax, linewidth=0.3, edgecolor="black")
ax.set_title(f"{ISO3} — Flood exposure: WorldPop + affected population (overlay)")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
fig.tight_layout()
fig.savefig(QC_PNG_PATH, dpi=200)
plt.close(fig)

print("Wrote QC figure:", QC_PNG_PATH)

run_metadata["flood_pop_affected"] = {
    "pop_tif": str(POP_TIF_PATH),
    "pop_weighted_days_tif": str(POP_WEIGHTED_DAYS_TIF),
    "qc_preview_png": str(QC_PNG_PATH),
    "worldpop_path": str(WORLDPOP_PATH),
    "worldpop_nodata": float(wp_nodata) if wp_nodata is not None else None,
    "worldpop_shape": [int(wp_shape[0]), int(wp_shape[1])],
    "worldpop_crs": str(wp_crs),
    "worldpop_bounds": [wp_bounds.left, wp_bounds.bottom, wp_bounds.right, wp_bounds.top],
    "days_path": str(flood_days_tif),
    "days_bounds": [
        float(flood_days_bounds.left),
        float(flood_days_bounds.bottom),
        float(flood_days_bounds.right),
        float(flood_days_bounds.top),
    ],
    "mask_path": str(out_tif),
    "days_aligned_to_worldpop": bool(aligned),
    "binary_threshold_days": int(FLOOD_BINARY_THRESHOLD_DAYS),
    "total_pop_valid_sum": total_pop,
    "pop_affected_sum": affected_pop,
    "pop_affected_pct": float(affected_pct) if np.isfinite(affected_pct) else None,
    "valid_pop_cells": valid_cells,
    "flooded_mask_cells": flooded_cells,
    "pop_affected_nonzero_cells": affected_cells,
    "max_flood_days": max_flood_days,
    "mean_flood_days_flooded_cells": mean_flood_days_flooded,
}
_write_metadata()

In [ ]:
# Cell 9 — Flood admin2 aggregation: binary exposure + flood-day severity metrics
# Output: one CSV row per adm2_pcode with population exposure and flood-day statistics

import json
import numpy as np
import pandas as pd
import rasterio
import rasterio.warp
from rasterio.features import rasterize
from pathlib import Path
from affine import Affine

# ---- Inputs ----
admin2 = admin2_gdf.copy()
if "adm2_pcode" not in admin2.columns:
    raise KeyError(f"'adm2_pcode' not found. Available columns: {list(admin2.columns)}")

POP_AFFECTED_FLOOD_TIF = (
    Path(DIRS["pop_exposed"])
    / "flood"
    / f"{ISO3}_pop_affected_flood_any_{WINDOW_START:%Y-%m-%d}_{WINDOW_END:%Y-%m-%d}.tif"
)
FLOOD_DAYS_TIF = (
    Path(DIRS["flood_native"])
    / "flood"
    / f"{ISO3}_flood_days_{WINDOW_START:%Y-%m-%d}_{WINDOW_END:%Y-%m-%d}.tif"
)

assert Path(WORLDPOP_PATH).exists(), f"Missing WorldPop: {WORLDPOP_PATH}"
assert POP_AFFECTED_FLOOD_TIF.exists(), f"Missing pop affected flood raster: {POP_AFFECTED_FLOOD_TIF}"
assert FLOOD_DAYS_TIF.exists(), f"Missing flood days raster: {FLOOD_DAYS_TIF}"

# ---- Load admin2 minimal columns ----
admin2 = (
    admin2[["adm2_pcode", "geometry"]]
    .dropna(subset=["adm2_pcode", "geometry"])
    .reset_index(drop=True)
)

# ---- Open WorldPop as reference grid ----
with rasterio.open(WORLDPOP_PATH) as wp:
    wp_arr = wp.read(1).astype("float32")
    wp_transform = wp.transform
    wp_crs = wp.crs
    wp_shape = (wp.height, wp.width)
    wp_nodata = wp.nodata

# Robust valid population mask
pop_valid = np.isfinite(wp_arr)
if wp_nodata is not None:
    pop_valid &= wp_arr != float(wp_nodata)
pop_valid &= wp_arr >= 0

# ---- Reproject admin2 to raster CRS and rasterize IDs ----
admin2_wp = admin2.to_crs(wp_crs)

shapes = [(geom, i + 1) for i, geom in enumerate(admin2_wp.geometry)]
admin2_id = rasterize(
    shapes=shapes,
    out_shape=wp_shape,
    transform=wp_transform,
    fill=0,
    dtype="int32",
    all_touched=True,
)

flat_id = admin2_id.ravel()
flat_wp = wp_arr.ravel()
flat_pop_valid = pop_valid.ravel()

valid_cells = (flat_id > 0) & flat_pop_valid
ids_valid = flat_id[valid_cells]
wp_valid_vals = flat_wp[valid_cells]

# Total population per admin2 id
pop_total_by_id = np.bincount(
    ids_valid, weights=wp_valid_vals, minlength=len(admin2_wp) + 1
)

# ---- Load flood pop affected raster and aggregate on same IDs ----
with rasterio.open(POP_AFFECTED_FLOOD_TIF) as src:
    aff_arr = src.read(1).astype("float32")
    aff_nodata = src.nodata

    # Grid alignment checks (tolerant transform comparison)
    same_crs = str(src.crs) == str(wp_crs)
    same_shape = (src.height == wp_shape[0]) and (src.width == wp_shape[1])

    same_transform = (
        isinstance(src.transform, Affine)
        and isinstance(wp_transform, Affine)
        and src.transform.almost_equals(wp_transform, precision=1e-9)
    )

    if not (same_crs and same_shape and same_transform):
        raise ValueError(
            "Flood pop-affected raster is not aligned to WorldPop grid.\n"
            f"WorldPop: crs={wp_crs}, shape={wp_shape}, transform={wp_transform}\n"
            f"Affected: crs={src.crs}, shape={(src.height, src.width)}, transform={src.transform}\n"
            f"Checks: same_crs={same_crs}, same_shape={same_shape}, same_transform={same_transform}"
        )

# Valid mask for affected raster (and require pop_valid so we never count nodata population)
aff_valid = np.isfinite(aff_arr)
if aff_nodata is not None:
    aff_valid &= aff_arr != float(aff_nodata)
aff_valid &= aff_arr >= 0

flat_aff = aff_arr.ravel()
flat_aff_valid = aff_valid.ravel()

valid_aff_cells = valid_cells & flat_aff_valid
ids_valid_aff = flat_id[valid_aff_cells]
aff_valid_vals = flat_aff[valid_aff_cells]

pop_aff_by_id = np.bincount(
    ids_valid_aff, weights=aff_valid_vals, minlength=len(admin2_wp) + 1
)


# ---- Load flood days raster and summarize severity by admin2 ----
with rasterio.open(FLOOD_DAYS_TIF) as src_days:
    days_arr = src_days.read(1).astype("float32")
    days_crs = src_days.crs
    days_shape = (src_days.height, src_days.width)
    days_transform = src_days.transform

    same_days_crs = str(days_crs) == str(wp_crs)
    same_days_shape = days_shape == wp_shape
    same_days_transform = (
        isinstance(days_transform, Affine)
        and isinstance(wp_transform, Affine)
        and days_transform.almost_equals(wp_transform, precision=1e-9)
    )
    days_aligned = bool(same_days_crs and same_days_shape and same_days_transform)

if not days_aligned:
    print("Flood days raster not aligned to WorldPop in Cell 9; reprojecting nearest for aggregation.")
    days_reproj = np.zeros(wp_shape, dtype="float32")
    rasterio.warp.reproject(
        source=days_arr,
        destination=days_reproj,
        src_transform=days_transform,
        src_crs=days_crs,
        dst_transform=wp_transform,
        dst_crs=wp_crs,
        resampling=rasterio.warp.Resampling.nearest,
        src_nodata=None,
        dst_nodata=0,
    )
    days_arr = days_reproj

flat_days = days_arr.ravel()
valid_days_cells = valid_cells & np.isfinite(flat_days)
ids_valid_days = flat_id[valid_days_cells]
days_valid_vals = flat_days[valid_days_cells]

# Mean flood days over valid-pop cells in each admin unit.
days_sum_by_id = np.bincount(ids_valid_days, weights=days_valid_vals, minlength=len(admin2_wp) + 1)
days_count_by_id = np.bincount(ids_valid_days, minlength=len(admin2_wp) + 1)

# Max flood days per admin via indexed maximum accumulation.
days_max_by_id = np.zeros(len(admin2_wp) + 1, dtype="float32")
np.maximum.at(days_max_by_id, ids_valid_days, days_valid_vals)

# ---- Assemble output table ----
n = len(admin2_wp)
admin2_ids = np.arange(1, n + 1, dtype=int)

out = pd.DataFrame(
    {
        "adm2_pcode": admin2_wp["adm2_pcode"].values,
        "pop_total": pop_total_by_id[admin2_ids].astype("float64"),
        "pop_affected_flood": pop_aff_by_id[admin2_ids].astype("float32"),
        "flood_days_mean": np.where(
            days_count_by_id[admin2_ids] > 0,
            days_sum_by_id[admin2_ids] / days_count_by_id[admin2_ids],
            np.nan,
        ).astype("float32"),
        "flood_days_max": days_max_by_id[admin2_ids].astype("float32"),
    }
)

out["pct_affected_flood"] = np.where(
    out["pop_total"] > 0,
    (out["pop_affected_flood"] / out["pop_total"]) * 100.0,
    np.nan,
)

# ---- Save CSV ----
OUT_CSV = (
    Path(DIRS["tables"])
    / f"{ISO3}_admin2_flood_exposure_{WINDOW_START:%Y-%m-%d}_{WINDOW_END:%Y-%m-%d}.csv"
)
out.to_csv(OUT_CSV, index=False)

print("Wrote:", OUT_CSV)
print(out.head(10).to_string(index=False))

# ---- Record metadata ----
run_metadata["admin2_flood_table"] = {
    "path": str(OUT_CSV),
    "n_admin2": int(len(out)),
    "worldpop_path": str(WORLDPOP_PATH),
    "pop_affected_flood_tif": str(POP_AFFECTED_FLOOD_TIF),
    "flood_days_tif": str(FLOOD_DAYS_TIF),
    "binary_threshold_days": int(FLOOD_BINARY_THRESHOLD_DAYS),
    "flood_days_aligned_to_worldpop": bool(days_aligned),
    "window_start": str(pd.to_datetime(WINDOW_START).date()),
    "window_end": str(pd.to_datetime(WINDOW_END).date()),
}
RUN_METADATA_PATH.write_text(json.dumps(run_metadata, indent=2, default=str))

In [ ]:
# Cell 10 — Flood QC pack: summary stats + figures + single PDF export
# Outputs:
#   - PNGs (WorldPop, flood mask, pop affected, admin2 choropleth)
#   - PDF QC pack combining key stats + all figures
#
# Assumes prior cells produced:
#   - out_tif (Cell 7 flood_any mask GeoTIFF)
#   - POP_TIF_PATH + QC_PNG_PATH (Cell 8 pop affected raster)
#   - OUT_CSV (Cell 9 admin2 exposure table)
#   - admin2_gdf (filtered to ISO3), WORLDPOP_PATH, DIRS, ISO3, WINDOW_START, WINDOW_END

import json
from pathlib import Path

import numpy as np
import pandas as pd
import rasterio
from rasterio.features import rasterize
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import matplotlib.patches as mpatches

# ---- Paths ----
QC_DIR = ensure_dir(DIRS["qc"] / "flood")
TABLE_PATH = (
    Path(DIRS["tables"])
    / f"{ISO3}_admin2_flood_exposure_{WINDOW_START:%Y-%m-%d}_{WINDOW_END:%Y-%m-%d}.csv"
)

MASK_DIR = ensure_dir(DIRS["flood_native"] / "flood")
FLOOD_MASK_TIF = MASK_DIR / f"{ISO3}_flood_any_{WINDOW_START:%Y-%m-%d}_{WINDOW_END:%Y-%m-%d}.tif"
FLOOD_DAYS_TIF = MASK_DIR / f"{ISO3}_flood_days_{WINDOW_START:%Y-%m-%d}_{WINDOW_END:%Y-%m-%d}.tif"
POP_AFFECTED_TIF = Path(POP_TIF_PATH)  # from Cell 8

FIG_WP = (
    QC_DIR
    / f"{ISO3}_flood_qc_worldpop_{WINDOW_START:%Y-%m-%d}_{WINDOW_END:%Y-%m-%d}.png"
)
FIG_MASK = (
    QC_DIR / f"{ISO3}_flood_qc_mask_{WINDOW_START:%Y-%m-%d}_{WINDOW_END:%Y-%m-%d}.png"
)
FIG_POP = (
    QC_DIR
    / f"{ISO3}_flood_qc_pop_affected_{WINDOW_START:%Y-%m-%d}_{WINDOW_END:%Y-%m-%d}.png"
)
FIG_DAYS = (
    QC_DIR
    / f"{ISO3}_flood_qc_days_{WINDOW_START:%Y-%m-%d}_{WINDOW_END:%Y-%m-%d}.png"
)
FIG_CHORO = (
    QC_DIR
    / f"{ISO3}_flood_qc_admin2_pct_{WINDOW_START:%Y-%m-%d}_{WINDOW_END:%Y-%m-%d}.png"
)

PDF_PATH = QC_DIR / f"{ISO3}_flood_QC_{WINDOW_START:%Y-%m-%d}_{WINDOW_END:%Y-%m-%d}.pdf"

assert WORLDPOP_PATH.exists(), f"Missing WorldPop: {WORLDPOP_PATH}"
assert FLOOD_MASK_TIF.exists(), f"Missing flood binary mask: {FLOOD_MASK_TIF}"
assert FLOOD_DAYS_TIF.exists(), f"Missing flood days raster: {FLOOD_DAYS_TIF}"
assert POP_AFFECTED_TIF.exists(), f"Missing pop affected: {POP_AFFECTED_TIF}"
assert TABLE_PATH.exists(), f"Missing admin2 table: {TABLE_PATH}"

# ---- Load admin2 + table ----
admin2 = admin2_gdf.copy()
if "adm2_pcode" not in admin2.columns:
    raise KeyError(f"'adm2_pcode' not found. Columns: {list(admin2.columns)}")

tbl = pd.read_csv(TABLE_PATH)
if "adm2_pcode" not in tbl.columns:
    raise KeyError(f"'adm2_pcode' missing from table. Columns: {list(tbl.columns)}")

# Join table back to geoms for choropleth
admin2_tbl = admin2.merge(tbl, on="adm2_pcode", how="left")

# ---- Load rasters ----
with rasterio.open(WORLDPOP_PATH) as wp:
    wp_arr = wp.read(1).astype("float32")
    wp_transform = wp.transform
    wp_crs = wp.crs
    wp_shape = (wp.height, wp.width)
    wp_nodata = wp.nodata
    wp_bounds = wp.bounds

# Valid mask for plotting/stats
wp_valid = np.isfinite(wp_arr)
if wp_nodata is not None:
    wp_valid &= wp_arr != float(wp_nodata)
wp_valid &= wp_arr >= 0

with rasterio.open(FLOOD_MASK_TIF) as fm:
    flood_mask = fm.read(1).astype("uint8")
    fm_nodata = fm.nodata
    fm_crs = fm.crs
    fm_transform = fm.transform

with rasterio.open(FLOOD_DAYS_TIF) as fd:
    flood_days = fd.read(1).astype("uint16")

with rasterio.open(POP_AFFECTED_TIF) as pa:
    pop_aff = pa.read(1).astype("float32")
    pa_nodata = pa.nodata
    pa_crs = pa.crs
    pa_transform = pa.transform

# ---- Basic sanity checks (should already align by design) ----
if (str(fm_crs) != str(wp_crs)) or (flood_mask.shape != wp_shape):
    print(("Flood mask raster not on WorldPop grid (CRS/shape mismatch)."))
    # raise ValueError("Flood mask raster not on WorldPop grid (CRS/shape mismatch).")

if (str(pa_crs) != str(wp_crs)) or (pop_aff.shape != wp_shape):
    print(("Pop-affected raster not on WorldPop grid (CRS/shape mismatch)."))
    # raise ValueError("Pop-affected raster not on WorldPop grid (CRS/shape mismatch).")

# ---- QC stats ----
total_pop = float(wp_arr[wp_valid].sum())
affected_pop = float(np.where(np.isfinite(pop_aff) & (pop_aff > 0), pop_aff, 0.0).sum())
affected_pct = (affected_pop / total_pop) * 100.0 if total_pop > 0 else np.nan

flooded_cells = int((flood_mask == 1).sum())
total_cells = int(flood_mask.size)
flooded_share = (flooded_cells / total_cells) * 100.0 if total_cells else np.nan
max_flood_days = int(flood_days.max())
mean_flood_days_flooded = float(flood_days[flood_mask == 1].mean()) if flooded_cells > 0 else 0.0

top10 = tbl.sort_values("pct_affected_flood", ascending=False).head(10)[
    ["adm2_pcode", "pop_total", "pop_affected_flood", "pct_affected_flood"]
]

print("QC summary:")
print(f"  WorldPop valid total pop: {total_pop:,.0f}")
print(f"  Flood-affected pop:       {affected_pop:,.0f} ({affected_pct:.3f}%)")
print(
    f"  Flooded cells (mask=1):   {flooded_cells:,} / {total_cells:,} ({flooded_share:.3f}%)"
)
print(f"  Flood days max (all cells): {max_flood_days}")
print(f"  Mean flood days on flooded cells: {mean_flood_days_flooded:.2f}")
print("  Top 10 admin2 by % affected:")
print(top10.to_string(index=False))


# ---- Plot helpers ----
def raster_extent(bounds):
    # bounds: left, bottom, right, top
    return (bounds.left, bounds.right, bounds.bottom, bounds.top)


extent = raster_extent(wp_bounds)

# Keep admin2 outlines thin black lines on all figures
admin2_outline = admin2.to_crs(wp_crs).boundary

# Downsample for PNG speed (does not affect data outputs)
DS = 8
wp_plot = np.where(wp_valid, wp_arr, np.nan)[::DS, ::DS]
mask_plot = np.where(flood_mask == 1, 1.0, np.nan)[::DS, ::DS]
days_plot = np.where(flood_days > 0, flood_days, np.nan)[::DS, ::DS]
pa_plot = np.where(pop_aff > 0, pop_aff, np.nan)[::DS, ::DS]

# ---- Figure 1: WorldPop (nodata-safe) ----
fig, ax = plt.subplots(figsize=(10, 8))
ax.imshow(wp_plot, extent=extent, origin="upper")
admin2_outline.plot(ax=ax, linewidth=0.3, edgecolor="black")
ax.set_title(f"{ISO3} — WorldPop (valid cells only)")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
fig.tight_layout()
fig.savefig(FIG_WP, dpi=200)
plt.close(fig)

# ---- Figure 2: Flood mask ----
fig, ax = plt.subplots(figsize=(10, 8))
ax.imshow(mask_plot, extent=extent, origin="upper")
admin2_outline.plot(ax=ax, linewidth=0.3, edgecolor="black")
ax.set_title(f"{ISO3} — Flood mask (any flood in window)")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
fig.tight_layout()
fig.savefig(FIG_MASK, dpi=200)
plt.close(fig)

# ---- Figure 3: Flood day severity ----
fig, ax = plt.subplots(figsize=(10, 8))
ax.imshow(days_plot, extent=extent, origin="upper")
admin2_outline.plot(ax=ax, linewidth=0.3, edgecolor="black")
ax.set_title(f"{ISO3} — Flood day severity (days flooded)")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
fig.tight_layout()
fig.savefig(FIG_DAYS, dpi=200)
plt.close(fig)

# ---- Figure 4: Population affected (overlay on WorldPop) ----
fig, ax = plt.subplots(figsize=(10, 8))
ax.imshow(wp_plot, extent=extent, origin="upper")
ax.imshow(pa_plot, extent=extent, origin="upper", alpha=0.65)
admin2_outline.plot(ax=ax, linewidth=0.3, edgecolor="black")
ax.set_title(f"{ISO3} — Population affected by flood (overlay on WorldPop)")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
fig.tight_layout()
fig.savefig(FIG_POP, dpi=200)
plt.close(fig)

# ---- Figure 5: Admin2 choropleth (% affected) ----
# Plot in EPSG:4326 for consistency; admin2_tbl already aligned by merge
admin2_tbl_4326 = admin2_tbl.to_crs("EPSG:4326")

fig, ax = plt.subplots(figsize=(10, 8))
admin2_tbl_4326.plot(
    column="pct_affected_flood",
    ax=ax,
    legend=True,
    missing_kwds={"color": "lightgrey", "label": "Missing"},
)
admin2_tbl_4326.boundary.plot(ax=ax, linewidth=0.3, edgecolor="black")
ax.set_title(f"{ISO3} — Admin2 % population affected by flood")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
fig.tight_layout()
fig.savefig(FIG_CHORO, dpi=200)
plt.close(fig)

print("Wrote QC PNGs:")
print(" ", FIG_WP)
print(" ", FIG_MASK)
print(" ", FIG_DAYS)
print(" ", FIG_POP)
print(" ", FIG_CHORO)

# ---- Build PDF QC pack ----
with PdfPages(PDF_PATH) as pdf:
    # Page 1: text summary
    fig, ax = plt.subplots(figsize=(8.27, 11.69))  # A4 portrait in inches-ish
    ax.axis("off")

    lines = []
    lines.append(f"Flood QC Pack — {ISO3}")
    lines.append(f"Window: {WINDOW_START:%Y-%m-%d} → {WINDOW_END:%Y-%m-%d}")
    lines.append("")
    lines.append("Key inputs")
    lines.append(f"  WorldPop: {WORLDPOP_PATH}")
    lines.append(f"  Flood binary mask: {FLOOD_MASK_TIF}")
    lines.append(f"  Flood days: {FLOOD_DAYS_TIF}")
    lines.append(f"  Pop affected: {POP_AFFECTED_TIF}")
    lines.append(f"  Admin2 table: {TABLE_PATH}")
    lines.append("")
    lines.append("QC summary")
    lines.append(f"  WorldPop nodata: {wp_nodata}")
    lines.append(f"  WorldPop valid total pop: {total_pop:,.0f}")
    lines.append(
        f"  Flooded cells (mask=1): {flooded_cells:,} / {total_cells:,} ({flooded_share:.3f}%)"
    )
    lines.append(f"  Flood days max: {max_flood_days}")
    lines.append(f"  Mean flood days on flooded cells: {mean_flood_days_flooded:.2f}")
    lines.append(f"  Flood-affected pop: {affected_pop:,.0f} ({affected_pct:.3f}%)")
    lines.append("")
    lines.append("Top 10 admin2 by % affected")
    for _, r in top10.iterrows():
        lines.append(
            f"  {r['adm2_pcode']}: {r['pct_affected_flood']:.2f}% "
            f"({r['pop_affected_flood']:,.0f} / {r['pop_total']:,.0f})"
        )

    ax.text(
        0.02,
        0.98,
        "\n".join(lines),
        va="top",
        ha="left",
        fontsize=10,
        family="monospace",
    )
    fig.tight_layout()
    pdf.savefig(fig)
    plt.close(fig)

    # Pages: figures
    for p in [FIG_WP, FIG_MASK, FIG_DAYS, FIG_POP, FIG_CHORO]:
        img = plt.imread(p)
        fig, ax = plt.subplots(figsize=(11.69, 8.27))  # A4 landscape
        ax.imshow(img)
        ax.axis("off")
        fig.tight_layout()
        pdf.savefig(fig)
        plt.close(fig)

print("Wrote QC PDF:", PDF_PATH)

# ---- Record to metadata ----
run_metadata["flood_qc_pack"] = {
    "qc_pngs": [str(FIG_WP), str(FIG_MASK), str(FIG_DAYS), str(FIG_POP), str(FIG_CHORO)],
    "qc_pdf": str(PDF_PATH),
    "summary": {
        "worldpop_total_pop_valid_sum": total_pop,
        "flooded_cells": flooded_cells,
        "mask_total_cells": total_cells,
        "flooded_share_pct": (
            float(flooded_share) if np.isfinite(flooded_share) else None
        ),
        "max_flood_days": max_flood_days,
        "mean_flood_days_flooded": mean_flood_days_flooded,
        "pop_affected_sum": affected_pop,
        "pop_affected_pct": float(affected_pct) if np.isfinite(affected_pct) else None,
    },
}
RUN_METADATA_PATH.write_text(json.dumps(run_metadata, indent=2, default=str))